# Лабораторная работа № 4

In [1]:
import math
import time
import tracemalloc
import numpy as np
import pandas as pd
from numba import njit
from IPython.display import display

asset_names = ("ЦБ1", "ЦБ2", "Депозит")
scenario_names = ("Благоприятная", "Нейтральная", "Негативная")
packages = np.array([25.0, 200.0, 100.0])
fees = np.array([0.04, 0.07, 0.05])
minimums = np.array([30.0, 150.0, 100.0])
probabilities = np.array([[0.60, 0.30, 0.10], [0.30, 0.20, 0.50], [0.40, 0.40, 0.20]])
market_growth = np.array([
    [[1.20, 1.10, 1.07], [1.05, 1.02, 1.03], [0.80, 0.95, 1.00]],
    [[1.40, 1.15, 1.01], [1.05, 1.00, 1.00], [0.60, 0.90, 1.00]],
    [[1.15, 1.12, 1.05], [1.05, 1.01, 1.01], [0.70, 0.94, 1.00]],
])
initial = np.array([100.0, 800.0, 400.0, 600.0])
initial_capital = float(initial.sum())
eps = 1e-8
expected_growth = np.einsum("ts,tsa->ta", probabilities, market_growth)

assert np.allclose(probabilities.sum(axis=1), 1.0)
assert np.all(packages > 0) and np.all(minimums >= 0)
assert np.all((fees >= 0) & (fees < 1))
assert np.all(market_growth >= 0)
print(f"Начальный капитал: {initial_capital:.2f} д.е.")
print("Размеры пакетов:", packages.tolist())
print("Комиссии:", fees.tolist())
print("Минимальные суммы:", minimums.tolist())

Начальный капитал: 1900.00 д.е.
Размеры пакетов: [25.0, 200.0, 100.0]
Комиссии: [0.04, 0.07, 0.05]
Минимальные суммы: [30.0, 150.0, 100.0]


In [2]:
market_rows = [
    {"Этап": stage + 1, "Ситуация": scenario_names[scenario],
     "Вероятность": probabilities[stage, scenario],
     **{asset_names[asset]: market_growth[stage, scenario, asset] for asset in range(3)}}
    for stage in range(3) for scenario in range(3)
]
display(pd.DataFrame(market_rows))
operation_rows = [
    {"Актив": asset_names[i], "Размер пакета": packages[i], "Комиссия": fees[i],
     "Покупка": packages[i] * (1 + fees[i]), "Продажа": packages[i] * (1 - fees[i]),
     "Минимум": minimums[i]}
    for i in range(3)
]
display(pd.DataFrame(operation_rows))

,Этап,Ситуация,Вероятность,ЦБ1,ЦБ2,Депозит
0,1,Благоприятная,0.6,1.20,1.10,1.07
1,1,Нейтральная,0.3,1.05,1.02,1.03
2,1,Негативная,0.1,0.80,0.95,1.00
3,2,Благоприятная,0.3,1.40,1.15,1.01
4,2,Нейтральная,0.2,1.05,1.00,1.00
5,2,Негативная,0.5,0.60,0.90,1.00
6,3,Благоприятная,0.4,1.15,1.12,1.05
7,3,Нейтральная,0.4,1.05,1.01,1.01
8,3,Негативная,0.2,0.70,0.94,1.00


,Актив,Размер пакета,Комиссия,Покупка,Продажа,Минимум
0,ЦБ1,25.0,0.04,26.0,24.0,30.0
1,ЦБ2,200.0,0.07,214.0,186.0,150.0
2,Депозит,100.0,0.05,105.0,95.0,100.0


## Математическая постановка задачи

портфель состоит из двух видов ценных бумаг, депозита и свободных денежных средств

Начальное состояние:

$$
S_1=(100,\ 800,\ 400,\ 600)
$$

Начальный капитал:

$$
K_0=100+800+400+600=1900
$$

На каждом этапе выбирается управление:

$$
u_t=(n_{t,1},n_{t,2},n_{t,3})
$$

где $n_{t,i}$ — целое количество пакетов актива $i$:

- $n_{t,i}>0$ — покупка
- $n_{t,i}<0$ — продажа
- $n_{t,i}=0$ — отсутствие сделки

Стоимость активов после операций:

$$
a'_{t,i}=a_{t,i}+q_i n_{t,i}
$$

где $q_i$ — фиксированный размер пакета

Комиссия за операции:

$$
C_t=\sum_{i=1}^{3}c_iq_i|n_{t,i}|
$$

Свободные деньги после операций:

$$
b'_t=b_t-\sum_{i=1}^{3}q_i n_{t,i}-C_t
$$

ограничения:

$$
a'_{t,i}\ge m_i,\qquad b'_t\ge0
$$

после сделок происходит одна из трёх рыночных ситуаций, в результате которой стоимость каждого актива умножается на соответствующий коэффициент роста

целевая функция — максимальный ожидаемый чистый доход:

$$
\max\mathbb E[K_4-K_0]
$$

где $K_4$ — итоговый капитал после третьего этапа


### Обоснование выбора критерия оптимальности

Задача рассматривается в условиях риска, поскольку для каждой рыночной ситуации известна вероятность её возникновения.

В качестве критерия оптимальности выбрано математическое ожидание итогового капитала. Оно учитывает как возможный финансовый результат каждого сценария, так и вероятность его реализации.

Критерий Вальда ориентирован на наихудший исход и применяется при выборе осторожной стратегии, когда основной целью является защита от максимальных потерь. В данной задаче требуется максимизировать средний ожидаемый финансовый результат, поэтому используется математическое ожидание.

Поскольку начальный капитал постоянен, максимизация ожидаемого чистого дохода эквивалентна максимизации ожидаемого итогового капитала.

$$
\max \mathbb E[K_4-K_0]
\quad\Longleftrightarrow\quad
\max \mathbb E[K_4]
$$



## Состояние инвестиционного портфеля

Состояние системы на каждом этапе задаётся четырьмя величинами:

$$
S_t=(a_{t,1},a_{t,2},a_{t,3},b_t)
$$

где:

- $a_{t,1}$ — стоимость первой ценной бумаги
- $a_{t,2}$ — стоимость второй ценной бумаги
- $a_{t,3}$ — сумма депозита
- $b_t$ — свободные денежные средства

Управление задаётся тройкой целых чисел:

$$
u_t=(n_1,n_2,n_3)
$$

Положительное число означает покупку пакетов, отрицательное— продажу, ноль— отсутствие операции

После выполнения сделок проверяются минимальные стоимости активов и неотрицательность свободных средств

Затем происходит одна из трёх рыночных ситуаций, определяющая состояние портфеля на следующем этапе


### Принятые трактовки условий задачи

1. Управление задаётся целым количеством покупаемых или продаваемых пакетов каждого актива.
2. Минимальная стоимость каждого актива проверяется после совершения сделок.
3. Рыночное падение может временно уменьшить стоимость актива ниже установленного минимума.
4. Если на следующем этапе актив находится ниже минимума, необходимо выбрать допустимое управление, восстанавливающее ограничение.
5. Продажи выполняются перед покупками. Средства, полученные от продажи, разрешается использовать для покупок на том же этапе.
6. Заёмные средства не используются.
7. Стратегия должна обеспечивать допустимое продолжение для каждого рыночного сценария с положительной вероятностью.

Эти правила используются при формировании допустимых управлений и вычислении функции Беллмана

In [3]:
@njit
def apply_trades(state, action):

    updated = state.copy()
    available_cash = state[3]

    for asset in range(3):
        if action[asset] >= 0:
            continue
        volume = -action[asset] * packages[asset]
        updated[asset] -= volume
        available_cash += volume * (1.0 - fees[asset])

    for asset in range(3):
        if action[asset] <= 0:
            continue
        volume = action[asset] * packages[asset]
        updated[asset] += volume
        available_cash -= volume * (1.0 + fees[asset])

    updated[3] = available_cash
    return updated


@njit
def is_allowed(state, action):
    updated = apply_trades(state, action)
    for asset in range(3):
        if updated[asset] < minimums[asset] - eps:
            return False
    return updated[3] >= -eps


@njit
def market_transition(state, stage, scenario):
    updated = state.copy()
    for asset in range(3):
        updated[asset] *= market_growth[stage, scenario, asset]
    return updated


@njit
def action_bounds(state):

    lower = np.empty(3, dtype=np.int64)
    upper = np.empty(3, dtype=np.int64)
    capital = np.sum(state)
    minimum_total = np.sum(minimums)

    for asset in range(3):
        package = packages[asset]
        lower[asset] = math.ceil((minimums[asset] - state[asset] - eps) / package)
        other_minimums = minimum_total - minimums[asset]
        available_for_asset = capital - other_minimums
        upper[asset] = math.floor((available_for_asset - state[asset] + eps) / package)
    return lower, upper


@njit
def terminal_capital(state):
    return np.sum(state)


@njit
def expected_last_capital(after_trades):
    total = after_trades[3]
    for asset in range(3):
        total += after_trades[asset] * expected_growth[2, asset]
    return total


example_action = np.array([1, 0, 0], dtype=np.int64)
example = apply_trades(initial, example_action)
assert np.allclose(example, [125, 800, 400, 574])
assert is_allowed(initial, example_action)
assert not is_allowed(
    np.array([49.0, 1012.0, 432.28, 746.0]),
    np.array([-1, 0, 0], dtype=np.int64),
)
print("Портфель после покупки одного пакета первой ценной бумаги:", example.tolist())

Портфель после покупки одного пакета первой ценной бумаги: [125.0, 800.0, 400.0, 574.0]



## Динамическое программирование и функция Беллмана

Для решения задачи используем принцип оптимальности Беллмана

Обозначим через $V_t(S)$ максимальный ожидаемый итоговый капитал, который можно получить, начиная с этапа $t$ из состояния портфеля $S$

На каждом этапе сначала выбирается допустимое управление $u$, затем реализуется один из трёх рыночных сценариев

Рекуррентное соотношение Беллмана:

$$
V_t(S)=\max_{u\in U(S)}
\sum_{j=1}^{3}p_{t,j}V_{t+1}(T_{t,j}(S,u))
$$

где:

- $U(S)$ — множество допустимых управлений
- $p_{t,j}$ — вероятность сценария $j$ на этапе $t$
- $T_{t,j}(S,u)$ — состояние после сделок и реализации сценария
- $V_{t+1}$ — оптимальный результат оставшихся этапов

Граничное условие:

$$
V_4(S)=K(S)
$$

где $K(S)$ — суммарный капитал портфеля после третьего этапа

После вычисления функций Беллмана оптимальные управления сохраняются для восстановления стратегии

In [4]:
growth_table = pd.DataFrame(expected_growth, columns=asset_names,
                            index=["Этап 1", "Этап 2", "Этап 3"])
display(growth_table.round(4))
margin_rows = []
for i in range(3):
    growth = expected_growth[2, i]
    margin_rows.append({
        "Актив": asset_names[i], "Ожидаемый рост": growth,
        "Эффект покупки пакета": packages[i] * (growth - 1 - fees[i]),
        "Эффект продажи пакета": packages[i] * (1 - fees[i] - growth),
    })
margin_table = pd.DataFrame(margin_rows)
display(margin_table.round(4))
assert (margin_table["Эффект покупки пакета"] < 0).all()
assert (margin_table["Эффект продажи пакета"] < 0).all()

,ЦБ1,ЦБ2,Депозит
Этап 1,1.115,1.061,1.051
Этап 2,0.930,0.995,1.003
Этап 3,1.020,1.040,1.024


,Актив,Ожидаемый рост,Эффект покупки пакета,Эффект продажи пакета
0,ЦБ1,1.020,-0.5,-1.5
1,ЦБ2,1.040,-6.0,-22.0
2,Депозит,1.024,-2.6,-7.4


### Обоснование расчёта последнего этапа и отсечения

На третьем этапе после решения остаётся ровно один рыночный переход. Поэтому ожидаемый капитал является линейной функцией сумм, вложенных в активы после сделок. Для пакета объёмом $q_i$, комиссии $c_i$ и ожидаемого коэффициента $\bar r_{3,i}$ изменение результата равно:

$$\Delta_{buy,i}=q_i(\bar r_{3,i}-1-c_i),\qquad
\Delta_{sell,i}=q_i(1-c_i-\bar r_{3,i}).$$

| Актив | Покупка | Продажа |
|---|---:|---:|
| ЦБ1 | −0,50 | −1,50 |
| ЦБ2 | −6,00 | −22,00 |
| Депозит | −2,60 | −7,40 |

Таким образом, **все добровольные сделки последнего этапа невыгодны**. Если значения активов уже выше минимумов, оптимально ничего не делать. Если минимум нарушен после изменения рынка, докупаем минимально необходимое число целых пакетов. Если свободных средств недостаточно, рассматриваем допустимые продажи других активов, начиная с небольшого объёма, достаточного для покрытия дефицита. Любая дополнительная продажа снижает ожидание и не нужна.

Для второго этапа при расчёте верхней оценки удобно сначала представить идеальный вариант, в котором после второго этапа отсутствуют издержки обязательных сделок третьего этапа. Тогда будущий рост каждого актива в среднем имеет множитель $g_i=\bar r_{2,i}\bar r_{3,i}=(0{,}9486;\ 1{,}0348;\ 1{,}027072)$. Поскольку $g_i<1+c_i$, добровольные покупки не могут улучшить результат такого расслабленного плана. Положительный вклад продаж учтён через $\max(0,1-c_i-g_i)$ с дополнительным ограничением объёма актива, доступного к продаже. **Верхняя оценка завышает, а не занижает достижимый результат**, так как игнорирует обязательные расходы и ограничения финансирования. Поэтому вариант можно отбросить, когда его верхняя оценка уже не выше найденного допустимого результата.

Обоснование опирается на указанные исходные коэффициенты и комиссии. При изменении параметров условия нужно проверять заново; без них используется полный перебор. Правило равенства в программе: обновление только при улучшении более чем на `EPS`. На первом этапе кандидаты рассматриваются после сортировки по убыванию верхней оценки.

In [5]:
@njit
def bellman_stage3(state):

    purchases = np.zeros(3, dtype=np.int64)
    purchase_cost = 0.0
    for asset in range(3):
        missing = max(0.0, minimums[asset] - state[asset])
        purchases[asset] = math.ceil((missing - eps) / packages[asset]) if missing > eps else 0
        purchase_cost += purchases[asset] * packages[asset] * (1.0 + fees[asset])


    if state[3] + eps >= purchase_cost:
        return expected_last_capital(apply_trades(state, purchases)), purchases

    cash_shortfall = purchase_cost - state[3]
    sale_limits = np.zeros(3, dtype=np.int64)
    for asset in range(3):
        if purchases[asset] != 0:
            continue
        sellable = max(0, math.floor((state[asset] - minimums[asset] + eps) / packages[asset]))
        proceeds_per_package = packages[asset] * (1.0 - fees[asset])
        enough_by_itself = math.ceil((cash_shortfall - eps) / proceeds_per_package)
        sale_limits[asset] = min(sellable, enough_by_itself)

    best_value = -np.inf
    best_action = np.zeros(3, dtype=np.int64)
    for first_sale in range(sale_limits[0] + 1):
        for second_sale in range(sale_limits[1] + 1):
            for third_sale in range(sale_limits[2] + 1):
                action = purchases - np.array([first_sale, second_sale, third_sale])
                if not is_allowed(state, action):
                    continue
                candidate = expected_last_capital(apply_trades(state, action))
                if candidate > best_value + eps:
                    best_value = candidate
                    best_action = action
    return best_value, best_action

@njit
def upper_bound_stage2(state):
    future = np.empty(3)
    for i in range(3):
        future[i] = expected_growth[1, i] * expected_growth[2, i]
    estimate = state[3]
    for i in range(3):
        estimate += state[i] * future[i]
        potential_sell = 1 - fees[i] - future[i]
        if potential_sell > 0:
            estimate += max(0.0, state[i] - minimums[i]) * potential_sell
    return estimate

@njit
def bellman_stage2(state):
    lower, upper = action_bounds(state)
    best = -np.inf
    best_action = np.zeros(3, dtype=np.int64)
    for i in range(lower[0], upper[0] + 1):
        for j in range(lower[1], upper[1] + 1):
            for k in range(lower[2], upper[2] + 1):
                action = np.array([i, j, k])
                if not is_allowed(state, action):
                    continue
                traded = apply_trades(state, action)
                if upper_bound_stage2(traded) <= best + eps:
                    continue
                value = 0.0
                for scenario in range(3):
                    following = market_transition(traded, 1, scenario)
                    continuation, _ = bellman_stage3(following)
                    if not np.isfinite(continuation):
                        value = -np.inf
                        break
                    value += probabilities[1, scenario] * continuation
                if value > best + eps:
                    best, best_action = value, action
    return best, best_action

@njit
def bellman_stage1(state):
    lower, upper = action_bounds(state)
    capacity = (upper[0] - lower[0] + 1) * (upper[1] - lower[1] + 1) * (upper[2] - lower[2] + 1)
    action_candidates = np.empty((capacity, 3), dtype=np.int64)
    candidate_bounds = np.empty(capacity)
    count = 0
    for i in range(lower[0], upper[0] + 1):
        for j in range(lower[1], upper[1] + 1):
            for k in range(lower[2], upper[2] + 1):
                action = np.array([i, j, k])
                if not is_allowed(state, action):
                    continue
                traded = apply_trades(state, action)
                bound = 0.0
                for scenario in range(3):
                    following = market_transition(traded, 0, scenario)
                    bound += probabilities[0, scenario] * upper_bound_stage2(following)
                action_candidates[count] = action
                candidate_bounds[count] = bound
                count += 1
    order = np.argsort(candidate_bounds[:count])[::-1]
    best = -np.inf
    best_action = np.zeros(3, dtype=np.int64)
    second_best = -np.inf
    second_action = np.zeros(3, dtype=np.int64)
    for index in order:
        if candidate_bounds[index] <= best + eps:
            break
        action = action_candidates[index]
        traded = apply_trades(state, action)
        value = 0.0
        for scenario in range(3):
            following = market_transition(traded, 0, scenario)
            continuation, _ = bellman_stage2(following)
            if not np.isfinite(continuation):
                value = -np.inf
                break
            value += probabilities[0, scenario] * continuation
        if value > best + eps:
            second_best, second_action = best, best_action.copy()
            best, best_action = value, action.copy()
        elif value > second_best + eps:
            second_best, second_action = value, action.copy()
    return best, best_action, count, second_best, second_action

print("Рекуррентные функции Беллмана для трёх этапов определены")

Рекуррентные функции Беллмана для трёх этапов определены


In [6]:
@njit
def exhaustive_last_stage(state):
    lower, upper = action_bounds(state)
    best = -np.inf
    for i in range(lower[0], upper[0] + 1):
        for j in range(lower[1], upper[1] + 1):
            for k in range(lower[2], upper[2] + 1):
                action = np.array([i, j, k])
                if is_allowed(state, action):
                    value = expected_last_capital(apply_trades(state, action))
                    best = max(best, value)
    return best

small_cases = [
    ("Без сделок", np.array([100., 800., 400., 600.]), np.array([0, 0, 0]), True),
    ("Покупка пакета", np.array([100., 800., 400., 600.]), np.array([1, 0, 0]), True),
    ("Нарушение минимума", np.array([49., 1012., 432.28, 746.]), np.array([-1, 0, 0]), False),
    ("Недостаток денег", np.array([30., 150., 100., 0.]), np.array([1, 0, 0]), False),
]
test_rows = []
for name, state, action, expected in small_cases:
    actual = bool(is_allowed(state, action))
    assert actual == expected
    test_rows.append({"Проверка": name, "Ожидается": expected, "Получено": actual, "Итог": "Пройдена"})
for name, state in [
    ("Все минимумы выполнены", np.array([50., 200., 100., 60.])),
    ("Необходима докупка", np.array([21., 200., 100., 60.])),
    ("Докупка за счёт продажи", np.array([21., 400., 100., 0.])),
    ("Недопустимое продолжение", np.array([21., 150., 100., 0.])),
]:
    fast, action = bellman_stage3(state)
    exhaustive = exhaustive_last_stage(state)
    actual = bool((np.isneginf(fast) and np.isneginf(exhaustive)) or abs(fast - exhaustive) < 1e-7)
    assert actual
    test_rows.append({"Проверка": name, "Ожидается": "Совпадение с полным перебором",
                      "Получено": "Совпадение" if actual else "Расхождение", "Итог": "Пройдена"})
display(pd.DataFrame(test_rows))

cash_flow_cases = [
    ("Продажа первой ценной бумаги", np.array([100.0, 800.0, 400.0, 600.0]),
     np.array([-1, 0, 0]), np.array([75.0, 800.0, 400.0, 624.0])),
    ("Покупка второй ценной бумаги", np.array([100.0, 800.0, 400.0, 600.0]),
     np.array([0, 1, 0]), np.array([100.0, 1000.0, 400.0, 386.0])),
    ("Продажа перед покупкой", np.array([100.0, 800.0, 400.0, 0.0]),
     np.array([-2, 0, 0]), np.array([50.0, 800.0, 400.0, 48.0])),
]
for label, start_state, action, expected_state in cash_flow_cases:
    actual_state = apply_trades(start_state, action)
    assert np.allclose(actual_state, expected_state), label
    assert is_allowed(start_state, action), label
    print(f"{label}: ожидалось {expected_state.tolist()}, получено {actual_state.tolist()}")


for action in (np.array([23, 0, 0]), np.array([19, 0, 1])):
    lower, upper = action_bounds(initial)
    assert np.all(action >= lower) and np.all(action <= upper)
    assert is_allowed(initial, action)



random_generator = np.random.default_rng(20261114)
random_test_count = 120
random_test_rows = []

for case_index in range(random_test_count):
    random_state = np.array([
        random_generator.uniform(0.0, 230.0),
        random_generator.uniform(0.0, 450.0),
        random_generator.uniform(0.0, 300.0),
        random_generator.uniform(0.0, 300.0),
    ])

    optimized_value, optimized_action = bellman_stage3(random_state)
    exhaustive_value = exhaustive_last_stage(random_state)

    values_match = (
        (np.isneginf(optimized_value) and np.isneginf(exhaustive_value))
        or np.isclose(
            optimized_value,
            exhaustive_value,
            atol=1e-7,
            rtol=0.0,
        )
    )

    assert values_match, (
        f"Несовпадение на третьем этапе в проверке {case_index + 1}: "
        f"состояние={random_state.tolist()}, "
        f"ускоренный расчёт={optimized_value}, "
        f"полный перебор={exhaustive_value}"
    )

    if np.isfinite(optimized_value):
        assert is_allowed(random_state, optimized_action)

    random_test_rows.append({
        "№": case_index + 1,
        "Входное состояние": np.array2string(
            random_state,
            precision=3,
            separator=", ",
        ),
        "Ожидалось (полный перебор)": exhaustive_value,
        "Получено (алгоритм)": optimized_value,
        "Результат": "Пройдена",
    })

random_test_results = pd.DataFrame(random_test_rows)

display(random_test_results)

print(
    f"Случайных проверок третьего этапа полным перебором "
    f"пройдено: {len(random_test_results)} из {random_test_count}"
)


,Проверка,Ожидается,Получено,Итог
0,Без сделок,True,True,Пройдена
1,Покупка пакета,True,True,Пройдена
2,Нарушение минимума,False,False,Пройдена
3,Недостаток денег,False,False,Пройдена
4,Все минимумы выполнены,Совпадение с полным перебором,Совпадение,Пройдена
5,Необходима докупка,Совпадение с полным перебором,Совпадение,Пройдена
6,Докупка за счёт продажи,Совпадение с полным перебором,Совпадение,Пройдена
7,Недопустимое продолжение,Совпадение с полным перебором,Совпадение,Пройдена


Продажа первой ценной бумаги: ожидалось [75.0, 800.0, 400.0, 624.0], получено [75.0, 800.0, 400.0, 624.0]
Покупка второй ценной бумаги: ожидалось [100.0, 1000.0, 400.0, 386.0], получено [100.0, 1000.0, 400.0, 386.0]
Продажа перед покупкой: ожидалось [50.0, 800.0, 400.0, 48.0], получено [50.0, 800.0, 400.0, 48.0]


,№,Входное состояние,Ожидалось (полный перебор),Получено (алгоритм),Результат
0,1,"[218.466, 74.826, 118.068, 58.823]",463.878445,463.878445,Пройдена
1,2,"[169.383, 422.094, 275.896, 3.268]",897.534160,897.534160,Пройдена
2,3,"[203.44 , 139.237, 44.713, 262.819]",647.820676,647.820676,Пройдена
3,4,"[105.721, 149.726, 0.565, 209.43 ]",-inf,-inf,Пройдена
4,5,"[ 26.394, 333.552, 59.834, 251.291]",683.276930,683.276930,Пройдена
...,...,...,...,...,...
115,116,"[111.49 , 91.974, 204.323, 240.811]",653.409680,653.409680,Пройдена
116,117,"[ 3.969, 171.276, 253.49 , 150.321]",591.069132,591.069132,Пройдена
117,118,"[ 78.335, 104.741, 166.064, 19.016]",-inf,-inf,Пройдена
118,119,"[157.262, 426.084, 210.319, 248.88 ]",1067.781963,1067.781963,Пройдена


Случайных проверок третьего этапа полным перебором пройдено: 120 из 120


In [7]:
from itertools import product

small_state = (50.0, 200.0, 100.0, 60.0)

def independent_step(state, action):
    updated = list(state)
    for index in range(3):
        amount = float(packages[index]) * action[index]
        updated[index] += amount
        updated[3] -= amount + float(fees[index]) * abs(amount)
    if min(updated[index] - float(minimums[index]) for index in range(3)) < -1e-8:
        return None
    if updated[3] < -1e-8:
        return None
    return tuple(updated)

def independent_last(state):
    capital = sum(state)
    ranges = []
    for index in range(3):
        package = float(packages[index])
        minimum = float(minimums[index])
        fee = float(fees[index])
        lower = math.ceil((minimum - state[index] - 1e-8) / package)
        upper = math.floor((capital - sum(float(x) for x in minimums) + minimum - state[index] + 1e-8) / package)
        upper = max(upper, 0)
        ranges.append(range(lower, upper + 1))
    best = -math.inf
    for action in product(*ranges):
        traded = independent_step(state, action)
        if traded is None:
            continue
        value = traded[3] + sum(traded[index] * float(expected_growth[2, index]) for index in range(3))
        best = max(best, value)
    return best

small_results = []
stage2_ranges = []
for index in range(3):
    package = float(packages[index])
    lower = math.ceil((float(minimums[index]) - small_state[index] - 1e-8) / package)
    upper = math.floor((sum(small_state) - sum(float(x) for x in minimums) + float(minimums[index]) - small_state[index] + 1e-8) / package)
    stage2_ranges.append(range(lower, upper + 1))
for action in product(*stage2_ranges):
    traded = independent_step(small_state, action)
    if traded is None:
        continue
    scenario_result = 0.0
    for scenario in range(3):
        state = tuple(traded[index] * float(market_growth[1, scenario, index]) for index in range(3)) + (traded[3],)
        continuation = independent_last(state)
        if not math.isfinite(continuation):
            scenario_result = -math.inf
            break
        scenario_result += float(probabilities[1, scenario]) * continuation
    small_results.append((action, scenario_result))
independent_value = max(score for action, score in small_results)
program_value, program_action = bellman_stage2(np.array(small_state))
assert math.isclose(independent_value, program_value, abs_tol=1e-7)
print("Независимый перебор всех допустимых управлений на малом портфеле")
print("Ожидаемое значение:", round(independent_value, 6))
print("Программное значение:", round(program_value, 6))
print("Результат: совпадение")

Независимый перебор всех допустимых управлений на малом портфеле
Ожидаемое значение: 417.0972
Программное значение: 417.0972
Результат: совпадение


In [8]:



from functools import lru_cache
from itertools import product

toy_probabilities = (0.5, 0.5)
toy_growth = (1.4, 0.8)
toy_price = 50.0
toy_initial = (0, 0.0, 100.0)


def toy_trade(state, action):
    units, asset, cash = state
    if action == 0:
        return state
    if action == 1 and units == 0 and cash >= toy_price:
        return (1, toy_price, cash - toy_price)
    return None


def toy_market(state, scenario):
    return (state[0], state[1] * toy_growth[scenario], state[2])


class IndependentPolicyEnumerator:


    def evaluate(self):
        histories = [()] + list(product(range(2), repeat=1)) + list(product(range(2), repeat=2))
        best_value = -math.inf
        best_policy = None
        feasible_count = 0
        for choices in product((0, 1), repeat=len(histories)):
            policy = dict(zip(histories, choices))
            branches = [((), toy_initial, 1.0)]
            for _ in range(3):
                next_branches = []
                for history, state, probability in branches:
                    traded = toy_trade(state, policy[history])
                    if traded is None:
                        break
                    for scenario, scenario_probability in enumerate(toy_probabilities):
                        next_branches.append((
                            history + (scenario,), toy_market(traded, scenario),
                            probability * scenario_probability,
                        ))
                else:
                    branches = next_branches
                    continue
                break
            else:
                feasible_count += 1
                value = sum(probability * (state[1] + state[2]) for _, state, probability in branches)
                if value > best_value:
                    best_value, best_policy = value, policy
        return best_value, best_policy, feasible_count


@lru_cache(maxsize=None)
def toy_bellman(stage, state):
    if stage == 3:
        return state[1] + state[2]
    values = []
    for action in (0, 1):
        traded = toy_trade(state, action)
        if traded is not None:
            values.append(sum(
                probability * toy_bellman(stage + 1, toy_market(traded, scenario))
                for scenario, probability in enumerate(toy_probabilities)
            ))
    return max(values)

brute_value, brute_policy, feasible_policies = IndependentPolicyEnumerator().evaluate()
bellman_value = toy_bellman(0, toy_initial)
assert math.isclose(brute_value, bellman_value, abs_tol=1e-9)
assert brute_policy[()] == 1
print("Независимая проверка трёхэтапной модели")
print("Допустимых адаптивных стратегий:", feasible_policies)
print("Ожидаемый капитал по полному перебору:", round(brute_value, 6))
print("Ожидаемый капитал по методу Беллмана:", round(bellman_value, 6))
print("Первое управление:", brute_policy[()])

assert feasible_policies > 1
assert math.isfinite(brute_value)
print("Область проверки: отдельная малая трёхэтапная модель; исходная модель не изменена")


Независимая проверка трёхэтапной модели
Допустимых адаптивных стратегий: 26
Ожидаемый капитал по полному перебору: 116.55
Ожидаемый капитал по методу Беллмана: 116.55
Первое управление: 1
Область проверки: отдельная малая трёхэтапная модель; исходная модель не изменена


In [9]:

class PortfolioInputValidator:


    @staticmethod
    def validate_state(state):
        values = np.asarray(state)
        if values.shape != (4,) or not np.issubdtype(values.dtype, np.number):
            raise ValueError("Состояние должно содержать четыре числовых значения")
        if not np.all(np.isfinite(values)) or np.any(values < 0):
            raise ValueError("Состояние должно содержать конечные неотрицательные значения")
        return values.astype(np.float64)

    @staticmethod
    def validate_action(action):
        values = np.asarray(action)
        if values.shape != (3,) or not np.issubdtype(values.dtype, np.number):
            raise ValueError("Управление должно содержать три числовых значения")
        if not np.all(np.isfinite(values)) or np.any(values != np.round(values)):
            raise ValueError("Управление должно содержать конечные целые количества пакетов")
        return values.astype(np.int64)

    @staticmethod
    def validate_parameters(probabilities, fees):
        probabilities = np.asarray(probabilities)
        fees = np.asarray(fees)
        if probabilities.shape != (3, 3) or fees.shape != (3,):
            raise ValueError("Неверные размерности параметров")
        if not np.all(np.isfinite(probabilities)) or np.any(probabilities < 0):
            raise ValueError("Некорректные вероятности")
        if not np.allclose(probabilities.sum(axis=1), 1.0):
            raise ValueError("Сумма вероятностей должна быть равна единице")
        if not np.all(np.isfinite(fees)) or np.any((fees < 0) | (fees >= 1)):
            raise ValueError("Некорректные комиссии")


validator = PortfolioInputValidator()
validator.validate_parameters(probabilities, fees)
invalid_cases = [
    ("Отрицательная стоимость актива", lambda: validator.validate_state([-1, 800, 400, 600])),
    ("Отрицательные свободные средства", lambda: validator.validate_state([100, 800, 400, -1])),
    ("Неопределённая стоимость актива", lambda: validator.validate_state([math.nan, 800, 400, 600])),
    ("Бесконечные свободные средства", lambda: validator.validate_state([100, 800, 400, math.inf])),
    ("Неверная длина состояния", lambda: validator.validate_state([100, 800, 600])),
    ("Дробное количество пакетов", lambda: validator.validate_action([1.5, 0, 0])),
    ("Неверная длина управления", lambda: validator.validate_action([1, 0])),
    ("Отрицательная вероятность", lambda: validator.validate_parameters(
        np.array([[-0.1, 0.6, 0.5], *probabilities[1:]]), fees)),
    ("Неверная сумма вероятностей", lambda: validator.validate_parameters(
        np.array([[0.5, 0.5, 0.5], *probabilities[1:]]), fees)),
    ("Отрицательная комиссия", lambda: validator.validate_parameters(
        probabilities, np.array([-0.1, 0.07, 0.05]))),
]
validation_rows = []
for name, check in invalid_cases:
    try:
        check()
        rejected = False
    except ValueError:
        rejected = True
    validation_rows.append({"Проверка": name, "Ожидается": "Отклонено", "Получено": "Отклонено" if rejected else "Принято", "Пройдено": rejected})


for name, state, action in [
    ("Недостаточно свободных средств", [30., 150., 100., 0.], [1, 0, 0]),
    ("Стоимость ниже минимума", [49., 1012., 432.28, 746.], [-1, 0, 0]),
]:
    accepted = bool(is_allowed(np.array(state), np.array(action, dtype=np.int64)))
    validation_rows.append({"Проверка": name, "Ожидается": "Отклонено", "Получено": "Принято" if accepted else "Отклонено", "Пройдено": not accepted})

validation_table = pd.DataFrame(validation_rows)
display(validation_table)
assert validation_table["Пройдено"].all()
print("Все проверки входных данных пройдены")


,Проверка,Ожидается,Получено,Пройдено
0,Отрицательная стоимость актива,Отклонено,Отклонено,True
1,Отрицательные свободные средства,Отклонено,Отклонено,True
2,Неопределённая стоимость актива,Отклонено,Отклонено,True
3,Бесконечные свободные средства,Отклонено,Отклонено,True
4,Неверная длина состояния,Отклонено,Отклонено,True
5,Дробное количество пакетов,Отклонено,Отклонено,True
6,Неверная длина управления,Отклонено,Отклонено,True
7,Отрицательная вероятность,Отклонено,Отклонено,True
8,Неверная сумма вероятностей,Отклонено,Отклонено,True
9,Отрицательная комиссия,Отклонено,Отклонено,True


Все проверки входных данных пройдены


In [10]:



from functools import lru_cache


def flat_market(state, scenario):
    return state


@lru_cache(maxsize=None)
def bellman_with_ties(stage, state):
    if stage == 3:
        return state[1] + state[2], ()

    best_value = -math.inf
    optimal_actions = []
    for action in (0, 1):
        traded = toy_trade(state, action)
        if traded is None:
            continue
        value = sum(
            probability * bellman_with_ties(
                stage + 1, flat_market(traded, scenario)
            )[0]
            for scenario, probability in enumerate(toy_probabilities)
        )
        if value > best_value + eps:
            best_value = value
            optimal_actions = [action]
        elif abs(value - best_value) <= eps:
            optimal_actions.append(action)
    return best_value, tuple(optimal_actions)


tied_value, tied_actions = bellman_with_ties(0, toy_initial)
assert math.isclose(tied_value, 100.0, abs_tol=1e-9)
assert tied_actions == (0, 1)
assert tied_actions[0] == 0
print("Проверка равных оптимумов: ожидается 100, получено ", tied_value)
print("Все оптимальные управления первого этапа:", tied_actions)
print("Выбранное управление (первое по порядку перебора):", tied_actions[0])
print("Область проверки: малая модель Беллмана; основной алгоритм Numba не изменён")


Проверка равных оптимумов: ожидается 100, получено  100.0
Все оптимальные управления первого этапа: (0, 1)
Выбранное управление (первое по порядку перебора): 0
Область проверки: малая модель Беллмана; основной алгоритм Numba не изменён


In [11]:
tracemalloc.start()
start = time.perf_counter()
optimal_capital, first_action, action_count, next_capital, next_action = bellman_stage1(initial)
elapsed = time.perf_counter() - start
memory_current, memory_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

assert np.isfinite(optimal_capital)
assert is_allowed(initial, first_action)
print("Количество допустимых управлений первого этапа:", action_count)
print("Оптимальное управление первого этапа:", tuple(first_action.tolist()))
print(f"Ожидаемый итоговый капитал: {optimal_capital:.5f} д.е.")
print(f"Ожидаемый чистый доход: {optimal_capital - initial_capital:.5f} д.е.")
print("Правило выбора при равенстве: первое вычисленное управление с максимальным значением")
print(f"Время выполнения с компиляцией: {elapsed:.3f} с")
print(f"Пиковая память Python, по tracemalloc: {memory_peak / 1024**2:.3f} МиБ")
print("Измерение tracemalloc не учитывает всю память нативного кода Numba")

Количество допустимых управлений первого этапа: 1502
Оптимальное управление первого этапа: (23, 0, 0)
Ожидаемый итоговый капитал: 2033.98888 д.е.
Ожидаемый чистый доход: 133.98888 д.е.
Правило выбора при равенстве: первое вычисленное управление с максимальным значением
Время выполнения с компиляцией: 16.718 с
Пиковая память Python, по tracemalloc: 12.218 МиБ
Измерение tracemalloc не учитывает всю память нативного кода Numba


In [12]:
@njit
def potentially_optimal_actions(state, threshold):
    lower, upper = action_bounds(state)
    capacity = (upper[0] - lower[0] + 1) * (upper[1] - lower[1] + 1) * (upper[2] - lower[2] + 1)
    candidates = np.empty((capacity, 3), dtype=np.int64)
    count = 0
    for i in range(lower[0], upper[0] + 1):
        for j in range(lower[1], upper[1] + 1):
            for k in range(lower[2], upper[2] + 1):
                action = np.array([i, j, k])
                if not is_allowed(state, action):
                    continue
                traded = apply_trades(state, action)
                bound = 0.0
                for scenario in range(3):
                    following = market_transition(traded, 0, scenario)
                    bound += probabilities[0, scenario] * upper_bound_stage2(following)
                if bound >= threshold - eps:
                    candidates[count] = action
                    count += 1
    return candidates[:count]

potential_actions = potentially_optimal_actions(initial, optimal_capital)
uniqueness_rows = []
for action in potential_actions:
    traded = apply_trades(initial, action)
    value = 0.0
    for scenario in range(3):
        following = market_transition(traded, 0, scenario)
        continuation, _ = bellman_stage2(following)
        value += probabilities[0, scenario] * continuation
    uniqueness_rows.append({"Управление": tuple(action.tolist()), "Ожидаемый капитал": value,
                            "Отставание": optimal_capital - value})
display(pd.DataFrame(uniqueness_rows).round(6))
number_of_optima = sum(abs(row["Отставание"]) < 1e-7 for row in uniqueness_rows)
print("Количество оптимальных управлений первого этапа:", number_of_optima)
print("Оптимум единственный" if number_of_optima == 1 else "Найдено несколько оптимальных управлений")

,Управление,Ожидаемый капитал,Отставание
0,"(15, 0, 2)",2033.691118,0.297766
1,"(19, 0, 1)",2033.872251,0.116633
2,"(23, 0, 0)",2033.988884,0.000000


Количество оптимальных управлений первого этапа: 1
Оптимум единственный


In [13]:
candidate_actions = [tuple(first_action.tolist()), (19, 0, 1), (0, 0, 0)]
manual_rows = []
for candidate in dict.fromkeys(candidate_actions):
    action = np.array(candidate, dtype=np.int64)
    if not is_allowed(initial, action):
        continue
    traded = apply_trades(initial, action)
    continuation_values = []
    for scenario in range(3):
        state = market_transition(traded, 0, scenario)
        value, follow_action = bellman_stage2(state)
        continuation_values.append(value)
    score = sum(probabilities[0, scenario] * continuation_values[scenario] for scenario in range(3))
    manual_rows.append({"Управление": candidate, "Первый благоприятный": continuation_values[0],
                        "Первый нейтральный": continuation_values[1],
                        "Первый негативный": continuation_values[2],
                        "Взвешенная оценка": score})
display(pd.DataFrame(manual_rows).round(5))
assert math.isclose(manual_rows[0]["Взвешенная оценка"], optimal_capital, abs_tol=1e-6)

traded_first = apply_trades(initial, first_action)
stage2_rows = []
for scenario in range(3):
    state = market_transition(traded_first, 0, scenario)
    value, action = bellman_stage2(state)
    stage2_rows.append({"Ситуация": scenario_names[scenario], "Состояние": tuple(np.round(state, 4)),
                        "Управление": tuple(action.tolist()), "Значение V₂": value})
display(pd.DataFrame(stage2_rows).round(5))

,Управление,Первый благоприятный,Первый нейтральный,Первый негативный,Взвешенная оценка
0,"(23, 0, 0)",2129.16182,1949.31571,1716.9708,2033.98888
1,"(19, 0, 1)",2122.80152,1953.32613,1741.9350,2033.87225
2,"(0, 0, 0)",2064.64782,1967.75846,1873.4848,2016.46471


,Ситуация,Состояние,Управление,Значение V₂
0,Благоприятная,"(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",2129.16182
1,Нейтральная,"(708.75, 816.0, 412.0, 2.0)","(-27, 0, 0)",1949.31571
2,Негативная,"(540.0, 760.0, 400.0, 2.0)","(-20, 0, 0)",1716.97080


In [14]:


stage2_calculation_rows = []
for first_scenario in range(3):
    state2 = market_transition(traded_first, 0, first_scenario)
    best_value, best_action = bellman_stage2(state2)
    alternatives = [best_action, np.zeros(3, dtype=np.int64)]
    for action in alternatives:
        if not is_allowed(state2, action):
            continue
        after_trades = apply_trades(state2, action)
        scenario_values = []
        stage3_decisions = []
        for second_scenario in range(3):
            state3 = market_transition(after_trades, 1, second_scenario)
            continuation, stage3_action = bellman_stage3(state3)
            scenario_values.append(float(continuation))
            stage3_decisions.append(tuple(stage3_action.tolist()))
        expected_value = sum(
            float(probabilities[1, k]) * scenario_values[k] for k in range(3)
        ) if all(map(math.isfinite, scenario_values)) else -math.inf
        stage2_calculation_rows.append({
            "Ситуация первого этапа": scenario_names[first_scenario],
            "Управление второго этапа": tuple(action.tolist()),
            "V3 при благоприятной": scenario_values[0],
            "V3 при нейтральной": scenario_values[1],
            "V3 при негативной": scenario_values[2],
            "V2 = 0.3*V3+ + 0.2*V30 + 0.5*V3-": expected_value,
            "Управления третьего этапа": tuple(stage3_decisions),
            "Отставание от оптимума": best_value - expected_value if math.isfinite(expected_value) else math.inf,
        })
        if np.array_equal(action, best_action):
            assert math.isclose(expected_value, best_value, abs_tol=1e-6)

stage2_calculation = pd.DataFrame(stage2_calculation_rows)
display(stage2_calculation.round(6))
for row in stage2_calculation_rows:
    if row["Ситуация первого этапа"] not in ("Нейтральная", "Негативная"):
        continue
    print(f"\n{row['Ситуация первого этапа']}, управление {row['Управление второго этапа']}:")
    print(
        f"V2 = 0.30 * {row['V3 при благоприятной']:.6f} "
        f"+ 0.20 * {row['V3 при нейтральной']:.6f} "
        f"+ 0.50 * {row['V3 при негативной']:.6f} "
        f"= {row['V2 = 0.3*V3+ + 0.2*V30 + 0.5*V3-']:.6f}"
    )

manual_terminal_rows = []
for first_scenario in (1, 2):
    state2 = market_transition(traded_first, 0, first_scenario)
    value2, action2 = bellman_stage2(state2)
    after_second_trades = apply_trades(state2, action2)
    for second_scenario in range(3):
        before_third_trades = market_transition(after_second_trades, 1, second_scenario)
        value3, action3 = bellman_stage3(before_third_trades)
        after_third_trades = apply_trades(before_third_trades, action3)
        direct_value = after_third_trades[3] + sum(
            after_third_trades[i] * sum(probabilities[2, k] * market_growth[2, k, i] for k in range(3))
            for i in range(3)
        )
        assert math.isclose(value3, direct_value, abs_tol=1e-6)
        manual_terminal_rows.append({
            "Ситуация после этапа 1": scenario_names[first_scenario],
            "Ситуация после этапа 2": scenario_names[second_scenario],
            "Состояние перед этапом 3": tuple(np.round(before_third_trades, 3)),
            "Управление третьего этапа": tuple(action3.tolist()),
            "Комиссия за сделки": round(sum(abs(action3[i]) * packages[i] * fees[i] for i in range(3)), 6),
            "Итоговый капитал по прямой формуле": direct_value,
        })
display(pd.DataFrame(manual_terminal_rows).round(6))

,Ситуация первого этапа,Управление второго этапа,V3 при благоприятной,V3 при нейтральной,V3 при негативной,V2 = 0.3*V3+ + 0.2*V30 + 0.5*V3-,Управления третьего этапа,Отставание от оптимума
0,Благоприятная,"(-31, 0, 0)",2291.11472,2136.95700,2028.872,2129.161816,"((0, 0, 0), (0, 0, 0), (1, 0, 0))",0.000
1,Благоприятная,"(0, 0, 0)",2653.81472,2222.98200,1759.672,2120.576816,"((0, 0, 0), (0, 0, 0), (0, 0, 0))",8.585
2,Нейтральная,"(-27, 0, 0)",2100.23788,1956.67425,1855.819,1949.315714,"((0, 0, 0), (0, 0, 0), (1, 0, 0))",0.000
3,Нейтральная,"(0, 0, 0)",2416.13788,2031.59925,1621.419,1941.870714,"((0, 0, 0), (0, 0, 0), (0, 0, 0))",7.445
4,Негативная,"(-20, 0, 0)",1861.77600,1724.84000,1626.940,1716.970800,"((0, 0, 0), (0, 0, 0), (1, 0, 0))",0.000
5,Негативная,"(0, 0, 0)",2095.77600,1780.34000,1453.440,1711.520800,"((0, 0, 0), (0, 0, 0), (0, 0, 0))",5.450



Нейтральная, управление (-27, 0, 0):
V2 = 0.30 * 2100.237880 + 0.20 * 1956.674250 + 0.50 * 1855.819000 = 1949.315714

Нейтральная, управление (0, 0, 0):
V2 = 0.30 * 2416.137880 + 0.20 * 2031.599250 + 0.50 * 1621.419000 = 1941.870714

Негативная, управление (-20, 0, 0):
V2 = 0.30 * 1861.776000 + 0.20 * 1724.840000 + 0.50 * 1626.940000 = 1716.970800

Негативная, управление (0, 0, 0):
V2 = 0.30 * 2095.776000 + 0.20 * 1780.340000 + 0.50 * 1453.440000 = 1711.520800


,Ситуация после этапа 1,Ситуация после этапа 2,Состояние перед этапом 3,Управление третьего этапа,Комиссия за сделки,Итоговый капитал по прямой формуле
0,Нейтральная,Благоприятная,"(47.25, 938.4, 416.12, 650.0)","(0, 0, 0)",0.0,2100.23788
1,Нейтральная,Нейтральная,"(35.438, 816.0, 412.0, 650.0)","(0, 0, 0)",0.0,1956.67425
2,Нейтральная,Негативная,"(20.25, 734.4, 412.0, 650.0)","(1, 0, 0)",1.0,1855.81900
3,Негативная,Благоприятная,"(56.0, 874.0, 404.0, 482.0)","(0, 0, 0)",0.0,1861.77600
4,Негативная,Нейтральная,"(42.0, 760.0, 400.0, 482.0)","(0, 0, 0)",0.0,1724.84000
5,Негативная,Негативная,"(24.0, 684.0, 400.0, 482.0)","(1, 0, 0)",1.0,1626.94000


In [15]:
branch_rows = []
for first_scenario in range(3):
    state2 = market_transition(traded_first, 0, first_scenario)
    stage2_value, stage2_action = bellman_stage2(state2)
    after_second = apply_trades(state2, stage2_action)
    for second_scenario in range(3):
        state3 = market_transition(after_second, 1, second_scenario)
        stage3_value, stage3_action = bellman_stage3(state3)
        after_third = apply_trades(state3, stage3_action)
        for third_scenario in range(3):
            last_state = market_transition(after_third, 2, third_scenario)
            branch_rows.append({
                "Ситуация 1": scenario_names[first_scenario],
                "Ситуация 2": scenario_names[second_scenario],
                "Ситуация 3": scenario_names[third_scenario],
                "Управление 1": tuple(first_action.tolist()),
                "Управление 2": tuple(stage2_action.tolist()),
                "Управление 3": tuple(stage3_action.tolist()),
                "Вероятность": float(probabilities[0, first_scenario] * probabilities[1, second_scenario] * probabilities[2, third_scenario]),
                "Итоговый капитал": float(terminal_capital(last_state)),
            })
branches = pd.DataFrame(branch_rows)
display(branches.round(5))
branch_probability = float(branches["Вероятность"].sum())
branch_expectation = float((branches["Вероятность"] * branches["Итоговый капитал"]).sum())
assert len(branches) == 27
assert math.isclose(branch_probability, 1.0, abs_tol=1e-8)
assert math.isclose(branch_expectation, optimal_capital, abs_tol=1e-6)
print(f"Сумма вероятностей: {branch_probability:.6f}")
print(f"Ожидаемый капитал по дереву из 27 исходов: {branch_expectation:.5f} д.е.")
print("Результат совпадает со значением функции Беллмана")

,Ситуация 1,Ситуация 2,Ситуация 3,Управление 1,Управление 2,Управление 3,Вероятность,Итоговый капитал
0,Благоприятная,Благоприятная,Благоприятная,"(23, 0, 0)","(-31, 0, 0)","(0, 0, 0)",0.072,2389.68400
1,Благоприятная,Благоприятная,Нейтральная,"(23, 0, 0)","(-31, 0, 0)","(0, 0, 0)",0.072,2256.17280
2,Благоприятная,Благоприятная,Негативная,"(23, 0, 0)","(-31, 0, 0)","(0, 0, 0)",0.036,2163.86000
3,Благоприятная,Нейтральная,Благоприятная,"(23, 0, 0)","(-31, 0, 0)","(0, 0, 0)",0.048,2223.26250
4,Благоприятная,Нейтральная,Нейтральная,"(23, 0, 0)","(-31, 0, 0)","(0, 0, 0)",0.048,2105.66750
5,Благоприятная,Нейтральная,Негативная,"(23, 0, 0)","(-31, 0, 0)","(0, 0, 0)",0.024,2026.92500
6,Благоприятная,Негативная,Благоприятная,"(23, 0, 0)","(-31, 0, 0)","(1, 0, 0)",0.120,2109.34000
7,Благоприятная,Негативная,Нейтральная,"(23, 0, 0)","(-31, 0, 0)","(1, 0, 0)",0.120,2000.50000
8,Благоприятная,Негативная,Негативная,"(23, 0, 0)","(-31, 0, 0)","(1, 0, 0)",0.060,1924.68000
9,Нейтральная,Благоприятная,Благоприятная,"(23, 0, 0)","(-27, 0, 0)","(0, 0, 0)",0.036,2192.27150


Сумма вероятностей: 1.000000
Ожидаемый капитал по дереву из 27 исходов: 2033.98888 д.е.
Результат совпадает со значением функции Беллмана


In [16]:
stage3_example = np.array([49.0, 1012.0, 432.28, 746.0])
invalid_action = np.array([-1, 0, 0])
print("Проверка ограничения на продажу ЦБ1")
print("Состояние:", stage3_example.tolist())
print("Управление:", tuple(invalid_action.tolist()))
print("Стоимость ЦБ1 после сделки:", apply_trades(stage3_example, invalid_action)[0])
print("Допустимость:", "Да" if is_allowed(stage3_example, invalid_action) else "Нет")
print("Вывод: продажа одного пакета ЦБ1 недопустима, так как 24 < 30")

Проверка ограничения на продажу ЦБ1
Состояние: [49.0, 1012.0, 432.28, 746.0]
Управление: (-1, 0, 0)
Стоимость ЦБ1 после сделки: 24.0
Допустимость: Нет
Вывод: продажа одного пакета ЦБ1 недопустима, так как 24 < 30


In [17]:

from itertools import product

analysis_rows = []
starting_capital = float(initial.sum())

for outcomes in product(range(3), repeat=3):
    state = initial.copy()
    branch_probability = 1.0
    record = {}
    total_fees = 0.0

    for stage, scenario in enumerate(outcomes):
        if stage == 0:
            action = first_action
        elif stage == 1:
            _, action = bellman_stage2(state)
        else:
            _, action = bellman_stage3(state)

        assert is_allowed(state, action)

        traded = apply_trades(state, action)
        fee = float(np.sum(np.abs(action) * packages * fees))
        total_fees += fee

        record[f"Управление {stage + 1}"] = tuple(int(x) for x in action)
        record[f"Комиссия {stage + 1}"] = fee
        record[f"После сделок {stage + 1}"] = tuple(
            round(float(x), 4) for x in traded
        )

        state = market_transition(traded, stage, scenario)

        record[f"После рынка {stage + 1}"] = tuple(
            round(float(x), 4) for x in state
        )

        branch_probability *= float(probabilities[stage, scenario])

    final_capital = float(terminal_capital(state))

    record.update({
        "Исход 1": scenario_names[outcomes[0]],
        "Исход 2": scenario_names[outcomes[1]],
        "Исход 3": scenario_names[outcomes[2]],
        "Вероятность": branch_probability,
        "Комиссии всего": total_fees,
        "Итоговый капитал": final_capital,
        "Чистый доход": final_capital - starting_capital,
        "Доходность, %": 100.0 * (
            final_capital / starting_capital - 1.0
        ),
    })

    analysis_rows.append(record)

financial_tree = pd.DataFrame(analysis_rows)

assert len(financial_tree) == 27
assert np.isclose(financial_tree["Вероятность"].sum(), 1.0)

expected_capital = float(np.dot(
    financial_tree["Вероятность"],
    financial_tree["Итоговый капитал"]
))

assert np.isclose(expected_capital, optimal_capital, atol=1e-6)

loss_probability = float(
    financial_tree.loc[
        financial_tree["Чистый доход"] < -eps,
        "Вероятность"
    ].sum()
)

minimum_capital = float(financial_tree["Итоговый капитал"].min())
maximum_capital = float(financial_tree["Итоговый капитал"].max())

expected_fees = float(np.dot(
    financial_tree["Вероятность"],
    financial_tree["Комиссии всего"]
))

print(
    "Полное дерево из 27 исходов. "
    "Состояния: (ЦБ1, ЦБ2, депозит, свободные средства)"
)

display(financial_tree.round(5))

print(f"Ожидаемый итоговый капитал: {expected_capital:.5f} д.е.")
print(
    f"Ожидаемый чистый доход: "
    f"{expected_capital - starting_capital:.5f} д.е."
)
print(
    f"Ожидаемая доходность: "
    f"{100.0 * (expected_capital / starting_capital - 1.0):.5f}%"
)
print(
    f"Вероятность убытка: {loss_probability:.6f} "
    f"({100.0 * loss_probability:.2f}%)"
)
print(f"Минимальный итоговый капитал: {minimum_capital:.5f} д.е.")
print(f"Максимальный итоговый капитал: {maximum_capital:.5f} д.е.")
print(f"Ожидаемая сумма комиссий: {expected_fees:.5f} д.е.")
print(
    "Комиссии уже учтены в итоговом капитале "
    "и повторно из прибыли не вычитаются"
)


Полное дерево из 27 исходов. Состояния: (ЦБ1, ЦБ2, депозит, свободные средства)


,Управление 1,Комиссия 1,После сделок 1,После рынка 1,Управление 2,Комиссия 2,После сделок 2,После рынка 2,Управление 3,Комиссия 3,После сделок 3,После рынка 3,Исход 1,Исход 2,Исход 3,Вероятность,Комиссии всего,Итоговый капитал,Чистый доход,"Доходность, %"
0,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",31.0,"(35.0, 880.0, 428.0, 746.0)","(49.0, 1012.0, 432.28, 746.0)","(0, 0, 0)",0.0,"(49.0, 1012.0, 432.28, 746.0)","(56.35, 1133.44, 453.894, 746.0)",Благоприятная,Благоприятная,Благоприятная,0.072,54.0,2389.68400,489.68400,25.77284
1,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",31.0,"(35.0, 880.0, 428.0, 746.0)","(49.0, 1012.0, 432.28, 746.0)","(0, 0, 0)",0.0,"(49.0, 1012.0, 432.28, 746.0)","(51.45, 1022.12, 436.6028, 746.0)",Благоприятная,Благоприятная,Нейтральная,0.072,54.0,2256.17280,356.17280,18.74594
2,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",31.0,"(35.0, 880.0, 428.0, 746.0)","(49.0, 1012.0, 432.28, 746.0)","(0, 0, 0)",0.0,"(49.0, 1012.0, 432.28, 746.0)","(34.3, 951.28, 432.28, 746.0)",Благоприятная,Благоприятная,Негативная,0.036,54.0,2163.86000,263.86000,13.88737
3,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",31.0,"(35.0, 880.0, 428.0, 746.0)","(36.75, 880.0, 428.0, 746.0)","(0, 0, 0)",0.0,"(36.75, 880.0, 428.0, 746.0)","(42.2625, 985.6, 449.4, 746.0)",Благоприятная,Нейтральная,Благоприятная,0.048,54.0,2223.26250,323.26250,17.01382
4,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",31.0,"(35.0, 880.0, 428.0, 746.0)","(36.75, 880.0, 428.0, 746.0)","(0, 0, 0)",0.0,"(36.75, 880.0, 428.0, 746.0)","(38.5875, 888.8, 432.28, 746.0)",Благоприятная,Нейтральная,Нейтральная,0.048,54.0,2105.66750,205.66750,10.82461
5,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",31.0,"(35.0, 880.0, 428.0, 746.0)","(36.75, 880.0, 428.0, 746.0)","(0, 0, 0)",0.0,"(36.75, 880.0, 428.0, 746.0)","(25.725, 827.2, 428.0, 746.0)",Благоприятная,Нейтральная,Негативная,0.024,54.0,2026.92500,126.92500,6.68026
6,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",31.0,"(35.0, 880.0, 428.0, 746.0)","(21.0, 792.0, 428.0, 746.0)","(1, 0, 0)",1.0,"(46.0, 792.0, 428.0, 720.0)","(52.9, 887.04, 449.4, 720.0)",Благоприятная,Негативная,Благоприятная,0.120,55.0,2109.34000,209.34000,11.01789
7,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",31.0,"(35.0, 880.0, 428.0, 746.0)","(21.0, 792.0, 428.0, 746.0)","(1, 0, 0)",1.0,"(46.0, 792.0, 428.0, 720.0)","(48.3, 799.92, 432.28, 720.0)",Благоприятная,Негативная,Нейтральная,0.120,55.0,2000.50000,100.50000,5.28947
8,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(810.0, 880.0, 428.0, 2.0)","(-31, 0, 0)",31.0,"(35.0, 880.0, 428.0, 746.0)","(21.0, 792.0, 428.0, 746.0)","(1, 0, 0)",1.0,"(46.0, 792.0, 428.0, 720.0)","(32.2, 744.48, 428.0, 720.0)",Благоприятная,Негативная,Негативная,0.060,55.0,1924.68000,24.68000,1.29895
9,"(23, 0, 0)",23.0,"(675.0, 800.0, 400.0, 2.0)","(708.75, 816.0, 412.0, 2.0)","(-27, 0, 0)",27.0,"(33.75, 816.0, 412.0, 650.0)","(47.25, 938.4, 416.12, 650.0)","(0, 0, 0)",0.0,"(47.25, 938.4, 416.12, 650.0)","(54.3375, 1051.008, 436.926, 650.0)",Нейтральная,Благоприятная,Благоприятная,0.036,50.0,2192.27150,292.27150,15.38271


Ожидаемый итоговый капитал: 2033.98888 д.е.
Ожидаемый чистый доход: 133.98888 д.е.
Ожидаемая доходность: 7.05205%
Вероятность убытка: 0.190000 (19.00%)
Минимальный итоговый капитал: 1533.26000 д.е.
Максимальный итоговый капитал: 2389.68400 д.е.
Ожидаемая сумма комиссий: 52.20000 д.е.
Комиссии уже учтены в итоговом капитале и повторно из прибыли не вычитаются
